In [5]:
"""
Find entity names that were extracted with more than one source_type/target_type
label across the resolved triples file, and generate an editable lookup table
for fixing them.

Why this happens: the closed 8-category taxonomy was finalized late in the
pipeline, after source_type/target_type had already been assigned per row.
Type assignment was validated per row, not per entity name across the whole
corpus, so the same resolved entity name can carry different type labels on
different rows -- e.g. "vacuum drying" assigned modification_method in some
rows and material_used_directly in others.

Writes three files:
  - entity_type_ambiguities.csv: read-only audit trail (one row per
    ambiguous name, full type breakdown, majority type/share, near-tie flag).
  - entity_type_ambiguities_context.json: example source sentences per
    (name, type), for judging each one.
  - entity_type_lookup.csv: THE EDITABLE LOOKUP TABLE. One row per
    ambiguous name with a resolved_type column, pre-filled with the
    majority-vote type. Edit resolved_type directly (by hand, or with
    apply_manual_llm_resolution.py / llm_resolve_entity_types.py) to fix an
    entity's type. build_kg_csvs.py reads this file.

Designed for Jupyter/Colab execution. No __main__ guard -- just set
TRIPLES_PATH below and run the whole cell/file.
"""

import re
import json
from collections import Counter, defaultdict

import pandas as pd

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = "final_formatting_cleaned_triples.xlsx"   # <- set this to your file

SOURCE_COL = "resolved_source"
TARGET_COL = "resolved_target"
SOURCE_TYPE_COL = "source_type"
TARGET_TYPE_COL = "target_type"
SENTENCE_COL = "corresponding_sentence"
DOI_COL = "doi"

NEAR_TIE_THRESHOLD = 0.65   # majority share below this is flagged as a near-tie
EXAMPLES_PER_TYPE = 4       # max example sentences kept per (entity, type) pair

OUT_AMBIGUITIES_CSV = "entity_type_ambiguities.csv"
OUT_CONTEXT_JSON = "entity_type_ambiguities_context.json"
OUT_LOOKUP_CSV = "entity_type_lookup.csv"

# ---------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------
if TRIPLES_PATH.lower().endswith(".csv"):
    df = pd.read_csv(TRIPLES_PATH)
else:
    df = pd.read_excel(TRIPLES_PATH)
print(f"Loaded {len(df)} triples from {TRIPLES_PATH}")

required = {SOURCE_COL, TARGET_COL, SOURCE_TYPE_COL, TARGET_TYPE_COL, SENTENCE_COL}
missing = required - set(df.columns)
if missing:
    raise SystemExit(f"STOP: input file is missing expected columns: {missing}")


def clean_text(val):
    if pd.isna(val):
        return ""
    s = str(val)
    s = re.sub(r"\s+", " ", s.replace("\n", " ").replace("\r", " ")).strip()
    return s


# ---------------------------------------------------------------
# TALLY TYPE LABELS PER ENTITY NAME (across both source and target roles)
# ---------------------------------------------------------------
name_type_counts = defaultdict(Counter)
name_type_examples = defaultdict(lambda: defaultdict(list))


def record(name_col, type_col):
    cols = [name_col, type_col, SENTENCE_COL, DOI_COL]
    for _, row in df[cols].dropna(subset=[name_col, type_col]).iterrows():
        name = row[name_col]
        t = row[type_col]
        name_type_counts[name][t] += 1
        bucket = name_type_examples[name][t]
        if len(bucket) < EXAMPLES_PER_TYPE:
            bucket.append({"sentence": clean_text(row[SENTENCE_COL]), "doi": clean_text(row[DOI_COL])})


record(SOURCE_COL, SOURCE_TYPE_COL)
record(TARGET_COL, TARGET_TYPE_COL)

ambiguous = {name: counter for name, counter in name_type_counts.items() if len(counter) > 1}
print(f"Entity names with conflicting type labels: {len(ambiguous)}")

# ---------------------------------------------------------------
# BUILD THE AUDIT TRAIL + LOOKUP TABLE + CONTEXT
# ---------------------------------------------------------------
ambiguity_rows = []
lookup_rows = []
context = {}

for name, counter in ambiguous.items():
    total = sum(counter.values())
    majority_type, majority_count = counter.most_common(1)[0]
    share = majority_count / total
    near_tie = share < NEAR_TIE_THRESHOLD
    breakdown = "; ".join(f"{t}={c}" for t, c in counter.most_common())

    ambiguity_rows.append({
        "entity_name": name,
        "total_occurrences": total,
        "type_breakdown": breakdown,
        "majority_type": majority_type,
        "majority_share": round(share, 3),
        "near_tie_flag": near_tie,
    })
    lookup_rows.append({
        "entity_name": name,
        "majority_type": majority_type,
        "majority_share": round(share, 3),
        "near_tie_flag": near_tie,
        "type_breakdown": breakdown,
        # This is the column build_kg_csvs.py actually reads. Edit it
        # directly to fix an entity's type -- it starts out equal to
        # majority_type until something (you, or an LLM script) changes it.
        "resolved_type": majority_type,
        "resolution_source": "majority_vote",
        "confidence": "",
        "rationale": "",
    })
    context[name] = {
        "type_breakdown": dict(counter),
        "examples_by_type": name_type_examples[name],
    }

ambig_df = pd.DataFrame(ambiguity_rows).sort_values(["near_tie_flag", "majority_share"], ascending=[False, True])
lookup_df = pd.DataFrame(lookup_rows).sort_values(["near_tie_flag", "majority_share"], ascending=[False, True])

print(f"  of which near-ties (majority share < {NEAR_TIE_THRESHOLD}): {int(ambig_df['near_tie_flag'].sum())}")

# ---------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------
ambig_df.to_csv(OUT_AMBIGUITIES_CSV, index=False)
lookup_df.to_csv(OUT_LOOKUP_CSV, index=False)
with open(OUT_CONTEXT_JSON, "w") as f:
    json.dump(context, f, indent=2)

print(f"\nWrote {OUT_AMBIGUITIES_CSV}")
print(f"Wrote {OUT_CONTEXT_JSON}")
print(f"Wrote {OUT_LOOKUP_CSV}  <- edit resolved_type here to fix an entity")

Loaded 10324 triples from final_formatting_cleaned_triples.xlsx
Entity names with conflicting type labels: 145
  of which near-ties (majority share < 0.65): 35

Wrote entity_type_ambiguities.csv
Wrote entity_type_ambiguities_context.json
Wrote entity_type_lookup.csv  <- edit resolved_type here to fix an entity


# testing to see if resolved

In [6]:
"""
Find entity names that were extracted with more than one source_type/target_type
label across the resolved triples file, and generate an editable lookup table
for fixing them.

Why this happens: the closed 8-category taxonomy was finalized late in the
pipeline, after source_type/target_type had already been assigned per row.
Type assignment was validated per row, not per entity name across the whole
corpus, so the same resolved entity name can carry different type labels on
different rows -- e.g. "vacuum drying" assigned modification_method in some
rows and material_used_directly in others.

Writes three files:
  - entity_type_ambiguities.csv: read-only audit trail (one row per
    ambiguous name, full type breakdown, majority type/share, near-tie flag).
  - entity_type_ambiguities_context.json: example source sentences per
    (name, type), for judging each one.
  - entity_type_lookup.csv: THE EDITABLE LOOKUP TABLE. One row per
    ambiguous name with a resolved_type column, pre-filled with the
    majority-vote type. Edit resolved_type directly (by hand, or with
    apply_manual_llm_resolution.py / llm_resolve_entity_types.py) to fix an
    entity's type. build_kg_csvs.py reads this file.

Designed for Jupyter/Colab execution. No __main__ guard -- just set
TRIPLES_PATH below and run the whole cell/file.
"""

import re
import json
from collections import Counter, defaultdict

import pandas as pd

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = "triples_types_resolved.xlsx"   # <- set this to your file

SOURCE_COL = "resolved_source"
TARGET_COL = "resolved_target"
SOURCE_TYPE_COL = "source_type"
TARGET_TYPE_COL = "target_type"
SENTENCE_COL = "corresponding_sentence"
DOI_COL = "doi"

NEAR_TIE_THRESHOLD = 0.65   # majority share below this is flagged as a near-tie
EXAMPLES_PER_TYPE = 4       # max example sentences kept per (entity, type) pair

OUT_AMBIGUITIES_CSV = "entity_type_ambiguities.csv"
OUT_CONTEXT_JSON = "entity_type_ambiguities_context.json"
OUT_LOOKUP_CSV = "entity_type_lookup.csv"

# ---------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------
if TRIPLES_PATH.lower().endswith(".csv"):
    df = pd.read_csv(TRIPLES_PATH)
else:
    df = pd.read_excel(TRIPLES_PATH)
print(f"Loaded {len(df)} triples from {TRIPLES_PATH}")

required = {SOURCE_COL, TARGET_COL, SOURCE_TYPE_COL, TARGET_TYPE_COL, SENTENCE_COL}
missing = required - set(df.columns)
if missing:
    raise SystemExit(f"STOP: input file is missing expected columns: {missing}")


def clean_text(val):
    if pd.isna(val):
        return ""
    s = str(val)
    s = re.sub(r"\s+", " ", s.replace("\n", " ").replace("\r", " ")).strip()
    return s


# ---------------------------------------------------------------
# TALLY TYPE LABELS PER ENTITY NAME (across both source and target roles)
# ---------------------------------------------------------------
name_type_counts = defaultdict(Counter)
name_type_examples = defaultdict(lambda: defaultdict(list))


def record(name_col, type_col):
    cols = [name_col, type_col, SENTENCE_COL, DOI_COL]
    for _, row in df[cols].dropna(subset=[name_col, type_col]).iterrows():
        name = row[name_col]
        t = row[type_col]
        name_type_counts[name][t] += 1
        bucket = name_type_examples[name][t]
        if len(bucket) < EXAMPLES_PER_TYPE:
            bucket.append({"sentence": clean_text(row[SENTENCE_COL]), "doi": clean_text(row[DOI_COL])})


record(SOURCE_COL, SOURCE_TYPE_COL)
record(TARGET_COL, TARGET_TYPE_COL)

ambiguous = {name: counter for name, counter in name_type_counts.items() if len(counter) > 1}
print(f"Entity names with conflicting type labels: {len(ambiguous)}")

# ---------------------------------------------------------------
# BUILD THE AUDIT TRAIL + LOOKUP TABLE + CONTEXT
# ---------------------------------------------------------------
ambiguity_rows = []
lookup_rows = []
context = {}

for name, counter in ambiguous.items():
    total = sum(counter.values())
    majority_type, majority_count = counter.most_common(1)[0]
    share = majority_count / total
    near_tie = share < NEAR_TIE_THRESHOLD
    breakdown = "; ".join(f"{t}={c}" for t, c in counter.most_common())

    ambiguity_rows.append({
        "entity_name": name,
        "total_occurrences": total,
        "type_breakdown": breakdown,
        "majority_type": majority_type,
        "majority_share": round(share, 3),
        "near_tie_flag": near_tie,
    })
    lookup_rows.append({
        "entity_name": name,
        "majority_type": majority_type,
        "majority_share": round(share, 3),
        "near_tie_flag": near_tie,
        "type_breakdown": breakdown,
        # This is the column build_kg_csvs.py actually reads. Edit it
        # directly to fix an entity's type -- it starts out equal to
        # majority_type until something (you, or an LLM script) changes it.
        "resolved_type": majority_type,
        "resolution_source": "majority_vote",
        "confidence": "",
        "rationale": "",
    })
    context[name] = {
        "type_breakdown": dict(counter),
        "examples_by_type": name_type_examples[name],
    }

ambig_df = pd.DataFrame(ambiguity_rows).sort_values(["near_tie_flag", "majority_share"], ascending=[False, True])
lookup_df = pd.DataFrame(lookup_rows).sort_values(["near_tie_flag", "majority_share"], ascending=[False, True])

print(f"  of which near-ties (majority share < {NEAR_TIE_THRESHOLD}): {int(ambig_df['near_tie_flag'].sum())}")

# ---------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------
ambig_df.to_csv(OUT_AMBIGUITIES_CSV, index=False)
lookup_df.to_csv(OUT_LOOKUP_CSV, index=False)
with open(OUT_CONTEXT_JSON, "w") as f:
    json.dump(context, f, indent=2)

print(f"\nWrote {OUT_AMBIGUITIES_CSV}")
print(f"Wrote {OUT_CONTEXT_JSON}")
print(f"Wrote {OUT_LOOKUP_CSV}  <- edit resolved_type here to fix an entity")

Loaded 10324 triples from triples_types_resolved.xlsx
Entity names with conflicting type labels: 0


KeyError: 'near_tie_flag'